In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import os 
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import parametros.load as load
import parametros.extract as extract
import parametros.fleet as fleet
import parametros.schedule as schedule
import parametros.metrics as metrics

In [3]:
TIPOS_DIAS = {
    "uteis": "307679",
    "sabado": "307681", 
    "domingo": "307680",
}

In [4]:
linhas_move = load.linhas_move

df = load.load_mco_table(r'./base_dados/mco_move', filtro_linhas=linhas_move, sep=';', encoding='latin1')
df = df[df['tipo dia'] == 8]

In [5]:
dict_gtfs = load.load_gtfs_tables(r'./base_dados/gtfsbhtrans', ['routes', 'trips', 'calendar', 'stop_times'])

In [6]:
trips = extract.get_trips_for_routes(dict_gtfs['trips'], dict_gtfs['routes'], route_short_names=linhas_move)

In [7]:
horarios = schedule.horarios_saida(trips, dict_gtfs['stop_times'],dict_service_id=TIPOS_DIAS)

Service_id não informado, considerando dia útil


In [8]:
headway = schedule.headway_por_hora(horarios)

In [9]:
headway

,hora,route_short_name,n_partidas,headway_min
0,0,10,3,20.0
1,0,51,3,20.0
2,0,5106,1,60.0
3,0,5201,1,60.0
4,0,5250,3,20.0
...,...,...,...,...
524,23,8101,3,20.0
525,23,82,2,30.0
526,23,8251,1,60.0
527,23,83P,2,30.0


In [10]:
frota_necessaria = fleet.frota_por_demanda_todas_linhas(
    headway,
    df
)

In [11]:
variabilidade_operacional = metrics.variabilidade_tempo_operacional(df, 'viagem', 'linha', 'saida', 'chegada')

In [12]:
df.columns

Index(['viagem', 'linha', 'sublinha', 'pc', 'concessionaria', 'saida',
       'veiculo', 'chegada', 'catraca saida', 'catraca chegada', 'ocorrencia',
       'justificativa', 'tipo dia', 'extensao', 'falha mecanica',
       'evento inseguro', 'indicador fechamento', 'data fechamento',
       'total usuarios', 'empresa operadora', 'unnamed: 20'],
      dtype='str')

In [ ]:
recarga = metrics.folga_recarga_garagem(
    dataframe=df[['linha', 'viagem','saida', 'chegada']],
    col_inicio='saida',
    col_fim='chegada',
    col_linha='linha',
    filtra_dia_mais_viagem=True,
    col_data = 'viagem'
)

In [ ]:
recarga